# Hand Action Detection Notebook

This notebook is part of a pipeline for hand pose sequence capture, dataset creation, model training, and inference. Comments and docs are concise and in English.


In [ ]:
# -------------------------
# 1 Imports and configuration
# -------------------------
import numpy as np
import json
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Dense, Dropout, GRU, TimeDistributed
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix

# -------------------------
# Load config from config.json
# -------------------------
with open('../config.json', 'r') as f:
    config = json.load(f)

# Common config
common_config = config['common']
ACTIONS = common_config['actions']
SEQUENCE_LENGTH = common_config['sequence_length']

# train_model-specific settings
train_config = config['train_model']
DATASET_PATH = train_config['dataset_path']
MODEL_EXPORT_NAME = train_config['model_export_name']
HAND_SELECTION = train_config['hand_selection']

# Model architecture settings
model_arch = train_config['model_architecture']
MODEL_TYPE = model_arch['type']
LAYERS_CONFIG = model_arch['layers']

# Training settings
training_config = train_config['training']
EPOCHS = training_config['epochs']
BATCH_SIZE = training_config['batch_size']
VALIDATION_SPLIT = training_config['validation_split']
OPTIMIZER = training_config['optimizer']
LOSS = training_config['loss']
METRICS = training_config['metrics']

# Data split settings
split_config = train_config['data_split']
TEST_SIZE = split_config['test_size']
STRATIFY = split_config['stratify']

# -------------------------
# 2 Load dataset from file
# -------------------------
data = np.load(DATASET_PATH)
X = data['X']  # forma original: (num_samples, sequence_length, 21, 3)
y_labels = data['y']

# Select hand(s) by configuration
if HAND_SELECTION == 'left':
    # Solo mano izquierda: primeros 21 landmarks
    X = X[:, :, 22:, :]
    print(f"Usando solo mano izquierda: {X.shape}")
elif HAND_SELECTION == 'right':
    # Solo mano derecha: últimos 21 landmarks
    X = X[:, :, :22, :]
    print(f"Usando solo mano derecha: {X.shape}")
else:  # 'both'
    # Ambas manos: mantener todos los 42 landmarks
    print(f"Usando ambas manos: {X.shape}")

# Aplanar landmarks de cada frame: (21,3) -> (63) o (42,3) -> (126)
num_samples = X.shape[0]
X = X.reshape(num_samples, SEQUENCE_LENGTH, -1)  # ahora (num_samples, sequence_length, features)

# Convertir labels a one-hot
y = to_categorical(y_labels, num_classes=len(ACTIONS))

# -------------------------
# 3️⃣ Train / Test split
# -------------------------
stratify_param = y_labels if STRATIFY else None
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=stratify_param, random_state=42
)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("y_train:", y_train.shape, "y_test:", y_test.shape)

# -------------------------
# 4️⃣ Crear modelo
# -------------------------
model = Sequential()

# Build layers based on configuration
for i, layer_config in enumerate(LAYERS_CONFIG):
    layer_type = layer_config['type']
    
    if layer_type == 'TimeDistributed_Dense':
        units = layer_config['units']
        activation = layer_config['activation']
        if i == 0:
            # Primera capa necesita input_shape
            model.add(TimeDistributed(Dense(units, activation=activation), 
                                     input_shape=(SEQUENCE_LENGTH, X.shape[2])))
        else:
            model.add(TimeDistributed(Dense(units, activation=activation)))
    
    elif layer_type == 'Dropout':
        rate = layer_config['rate']
        model.add(Dropout(rate))
    
    elif layer_type == 'GRU':
        units = layer_config['units']
        return_sequences = layer_config.get('return_sequences', False)
        model.add(GRU(units, return_sequences=return_sequences))
    
    elif layer_type == 'Dense':
        units = layer_config.get('units', len(ACTIONS))
        activation = layer_config['activation']
        model.add(Dense(units, activation=activation))

model.compile(
    optimizer=OPTIMIZER,
    loss=LOSS,
    metrics=METRICS
)

model.summary()

# -------------------------
# 5 Training
# -------------------------
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT,
    batch_size=BATCH_SIZE,
    verbose=1
)

# Save trained model
model.save(MODEL_EXPORT_NAME)
print(f"Modelo guardado en {MODEL_EXPORT_NAME}")

# -------------------------
# 6 Evaluation / Metrics
# -------------------------
model = load_model(MODEL_EXPORT_NAME)

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)

acc = accuracy_score(y_true, y_pred)
cm = confusion_matrix(y_true, y_pred)

print(f"Accuracy en test set: {acc:.4f}")
print("Matriz de confusión:")
print(cm)
